# Model Comparison

Compare five classifiers on the engineered dataset using stratified 5-fold cross-validation, then evaluate the best candidates on a held-out test set with confusion matrices and ROC curves.

> **Prerequisite**: Run `03_feature_engineering.ipynb` first to generate `data/student-mat-engineered.csv`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from src.data import DATA_DIR
from src.preprocessing import build_preprocessor
from src.train import cross_validate_models
from src.evaluate import plot_confusion_matrix, plot_roc_curve, print_report

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Load Data & Prepare

In [ ]:
df = pd.read_csv(DATA_DIR / "student-mat-engineered.csv")

X = df.drop(columns=["at_risk"])
y = df["at_risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target balance (train): {y_train.value_counts().to_dict()}")

## 2. Cross-Validation (5 Models)

In [ ]:
preprocessor = build_preprocessor(X_train)

models = {
    "Logistic Regression": Pipeline([
        ("pre", preprocessor),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    ]),
    "Random Forest": Pipeline([
        ("pre", preprocessor),
        ("clf", RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)),
    ]),
    "XGBoost": Pipeline([
        ("pre", preprocessor),
        ("clf", XGBClassifier(
            n_estimators=200, use_label_encoder=False, eval_metric="logloss",
            scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
            random_state=42, verbosity=0,
        )),
    ]),
    "LightGBM": Pipeline([
        ("pre", preprocessor),
        ("clf", LGBMClassifier(
            n_estimators=200, class_weight="balanced", random_state=42, verbose=-1,
        )),
    ]),
    "SVM": Pipeline([
        ("pre", preprocessor),
        ("clf", SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=42)),
    ]),
}

In [ ]:
results = cross_validate_models(models, X_train, y_train, cv=5)
results = results.sort_values("f1_mean", ascending=False).reset_index(drop=True)
results

## 3. SMOTE Alternative

Test SMOTE oversampling (applied only during training via `imblearn.pipeline`).

In [ ]:
smote_models = {
    "LR + SMOTE": ImbPipeline([
        ("pre", preprocessor),
        ("smote", SMOTE(random_state=42)),
        ("clf", LogisticRegression(max_iter=1000, random_state=42)),
    ]),
    "RF + SMOTE": ImbPipeline([
        ("pre", preprocessor),
        ("smote", SMOTE(random_state=42)),
        ("clf", RandomForestClassifier(n_estimators=200, random_state=42)),
    ]),
}

smote_results = cross_validate_models(smote_models, X_train, y_train, cv=5)
smote_results

## 4. Comparison Chart

In [ ]:
all_results = pd.concat([results, smote_results], ignore_index=True)
all_results = all_results.sort_values("f1_mean", ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(all_results["model"], all_results["f1_mean"], xerr=all_results["f1_std"],
               color="steelblue", edgecolor="black", capsize=4)
ax.set_xlabel("Mean F1 Score (5-fold CV)")
ax.set_title("Model Comparison — F1 Score")
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

## 5. Test-Set Evaluation (Confusion Matrices & ROC Curves)

In [ ]:
# Fit all models on full training set and evaluate on test set
fig_cm, axes_cm = plt.subplots(1, len(models), figsize=(5 * len(models), 4))
fig_roc, ax_roc = plt.subplots(figsize=(8, 6))

for i, (name, model) in enumerate(models.items()):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Confusion matrix
    plot_confusion_matrix(y_test, y_pred, title=name, ax=axes_cm[i])

    # ROC curve (all on same axes)
    plot_roc_curve(model, X_test, y_test, title=name, ax=ax_roc)

fig_cm.suptitle("Confusion Matrices (Test Set)", fontsize=14, y=1.02)
fig_cm.tight_layout()

ax_roc.set_title("ROC Curves (Test Set)")
ax_roc.legend(loc="lower right")
fig_roc.tight_layout()

plt.show()

In [ ]:
for name, model in models.items():
    y_pred = model.predict(X_test)
    print(f"=== {name} ===")
    print_report(y_test, y_pred)
    print()